# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule (in words):** score every page higher if it's (a) visible — lots of impressions, (b) stale — a long time since it was last updated, (c) sitting in a position where a small push could matter, and a little bit for (d) being thin relative to how visible it already is. Weighted 40% visibility, 30% staleness, 25% position opportunity, 5% depth gap.

**Reason codes it can output:** `stale_visible_page`, `declining_with_demand`, `thin_visible_page`, `page_one_decay_risk`, `low_ctr_visible_page`, `low_engagement_visible_page`, and `general_refresh_review` as the fallback when none of the specific conditions fire.

In [1]:
print("baseline_refresh_score = 0.40*visibility + 0.30*staleness + 0.25*position_opportunity + 0.05*depth_gap")
print()
print("Reason codes: stale_visible_page, declining_with_demand, thin_visible_page,")
print("              page_one_decay_risk, low_ctr_visible_page, low_engagement_visible_page,")
print("              general_refresh_review (fallback)")


baseline_refresh_score = 0.40*visibility + 0.30*staleness + 0.25*position_opportunity + 0.05*depth_gap

Reason codes: stale_visible_page, declining_with_demand, thin_visible_page,
              page_one_decay_risk, low_ctr_visible_page, low_engagement_visible_page,
              general_refresh_review (fallback)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import sys, subprocess
import pandas as pd

subprocess.run([sys.executable, "../../scripts/02_baseline_score.py"], check=True)

queue = pd.read_csv("../../data/processed/baseline_refresh_queue.csv")
print(f"Ranked {len(queue):,} pages")
print(f"Top-50 declining rate (baseline rule): {queue.head(50)['is_declining_label'].mean():.1%}")

import os
os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")


Wrote baseline queue: /home/claude/repo/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340
Ranked 30,000 pages
Top-50 declining rate (baseline rule): 34.0%


Wrote work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Confidence note: the baseline is a deterministic rule, not a probabilistic model, so "confidence" here means how many reason codes fired at once — a page flagged for 2-3 reasons is a firmer pick than one that only hit `general_refresh_review`. What would make a top-20 pick wrong: a page that's visible and stale by the numbers but was *intentionally* left unchanged (a reference page, a legal/policy page) — the rule has no way to know intent.

In [3]:
cols = ["baseline_rank", "reason_codes", "suggested_action_baseline", "is_declining_label", "impressions_90d"]
top20 = queue.sort_values("baseline_rank").head(20)[cols]
top20["n_reasons"] = queue.sort_values("baseline_rank").head(20)["reason_codes"].str.count("\\|") + 1
top20


,baseline_rank,reason_codes,suggested_action_baseline,is_declining_label,impressions_90d,n_reasons
0,1,declining_with_demand|page_one_decay_risk|low_...,refresh,1,309192,3
1,2,page_one_decay_risk|low_engagement_visible_page,monitor,0,97999,2
2,3,page_one_decay_risk|low_engagement_visible_page,monitor,0,101078,2
3,4,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0,117741,3
4,5,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0,152617,3
5,6,declining_with_demand|page_one_decay_risk|low_...,refresh_and_review_ctr,1,145292,4
6,7,page_one_decay_risk|low_engagement_visible_page,monitor,0,79146,2
7,8,page_one_decay_risk|low_engagement_visible_page,monitor,0,142072,2
8,9,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0,148737,3
9,10,page_one_decay_risk|low_engagement_visible_page,monitor,0,129239,2


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

A "weak" pick in the top 20 is one where `is_declining_label` is 0 — the rule flagged it on visibility/staleness/position, but it wasn't actually trending down over the last 30 days. That's expected and fine: the baseline rule doesn't see the label at all (it's a set of hand-written thresholds on visibility, staleness, and position), so a non-declining page showing up just means the page is a genuine stale-but-stable candidate for review, not a rule failure.

In [4]:
top20_full = queue.sort_values("baseline_rank").head(20)
weak = top20_full[top20_full["is_declining_label"] == 0]
print(f"Weak picks in top 20 (flagged but not labeled declining): {len(weak)} of 20")
print()

# Leakage check: confirm the score formula itself never touches trend_direction / trend_pct
src = open("../../scripts/02_baseline_score.py").read()
formula_block = src.split("def reason_codes")[0]
print("trend_pct used in scoring formula   :", "trend_pct" in formula_block)
print("trend_direction used in the formula :", "trend_direction" in formula_block)
print("(trend_direction is only read later, to WRITE a reason code — it never feeds baseline_refresh_score)")


Weak picks in top 20 (flagged but not labeled declining): 13 of 20

trend_pct used in scoring formula   : False
trend_direction used in the formula : False
(trend_direction is only read later, to WRITE a reason code — it never feeds baseline_refresh_score)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.